# Miscellaneous plots of met forcing transport simulations

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from min3p.output import read_min3p, read_min3p_sequence
from byte_util.util import all_sites
from tqdm.notebook import tqdm

all_scenarios = ['longterm', 'monthly', 'daily', 'hourly']

In [ ]:
site = 'Cecil'
fig, ax = plt.subplots(1, 4, figsize=(8, 6), sharey=True)

sim_folder = Path(f'../min3p_runs/{site}/spinup')
sim_name = 'spinup'

gsp, gsp_cols, timesteps = read_min3p_sequence(f'{sim_name}_1.gsp', folder=sim_folder)

gsc, gsc_cols, timesteps = read_min3p_sequence(f'{sim_name}_1.gsc', folder=sim_folder)

cmap = plt.colormaps['cividis']
colors = cmap(np.linspace(0, 1, gsp.shape[0]))

for i, timestep in enumerate(timesteps):

    z = gsp[i, gsp_cols.index('z'), :]
    depth = z.max() - z  # Convert z (elevation) to depth

    label = '%d d' % round(timestep)
    vars_to_plot = ['p_w', 'theta_a', 'q_root']
    for j, col in enumerate(vars_to_plot):
        ax[j].plot(gsp[i, gsp_cols.index(col), :], depth, color=colors[i], label=label)
        ax[j].set(xlabel=col)
    ax[-1].plot(gsc[i, gsc_cols.index('psi01'), ], depth, color=colors[i])

ax[-1].set(xlabel='Tracer (mol/L)')
ax[0].set(ylabel='Depth', ylim=(depth.max(), depth.min()))

ax[0].legend()
ax[1].set(title=site)

## Plot soil moisture across variable forcing period

In [ ]:
control_planes_to_plot = [1, 5, 10, 30,
                          50, 100, 200, 300]

for site in tqdm(all_sites):

    fig, ax = plt.subplots(2, 4, figsize=(10, 7), sharex='all', sharey='all')

    for i, scenario in enumerate(all_scenarios):
        sim_folder = Path(f'../min3p_runs/{site}/{scenario}')
        sim_name = scenario

        gbp, gbp_vars, grid_cells = read_min3p_sequence(f'{sim_name}_1.gbp',
                                                   folder=sim_folder, ftype='transient')
        for j, cp in enumerate(control_planes_to_plot):
            grid_cell = 401 - cp # Convert from depth to grid cell number
            idx = list(grid_cells).index(grid_cell)
            ax.flatten()[j].plot(gbp[idx, gbp_vars.index('time')],
                                 gbp[idx, gbp_vars.index('theta_a')], lw=1, label=scenario)
            depth = f'depth={cp} cm'
            ax.flatten()[j].set(title=depth, xlim=[0, 3650])
            j += 1

    ax[-1, -1].legend()
    ax[0, 0].set(ylabel='Water content (m3/m3)')
    ax[1, 0].set(ylabel='Water content (m3/m3)')
    fig.suptitle(site)

    for i in range(4):
        ax[1, i].set(xlabel='Time (d)')

    fig.savefig(f'plots/{site}_VariableWaterContent.png', dpi=300, bbox_inches='tight')

    # Adjust the axis limits and save again
    ax[0, 0].set(xlim=[365, 730])
    fig.savefig(f'plots/{site}_VariableWaterContent_365days.png', dpi=300, bbox_inches='tight')

    # Adjust the axis limits and save again
    ax[0, 0].set(xlim=[365, 455])
    fig.savefig(f'plots/{site}_VariableWaterContent_90days.png', dpi=300, bbox_inches='tight')

    if site != 'Cecil':
        plt.close(fig)

In [ ]:
for site in tqdm(all_sites):

    fig, ax = plt.subplots(2, 4, figsize=(10, 7), sharex='all', sharey='all')

    for i, scenario in enumerate(all_scenarios):
        sim_folder = Path(f'../min3p_runs/{site}/{scenario}')
        sim_name = scenario

        gbp, gbp_vars, grid_cells = read_min3p_sequence(f'{sim_name}_1.gbp',
                                                   folder=sim_folder, ftype='transient')
        for j, cp in enumerate(control_planes_to_plot):
            grid_cell = 401 - cp # Convert from depth to grid cell number
            idx = list(grid_cells).index(grid_cell)
            # q_root is in m3/d, but given that dx=1, dy=1 this is effectively m/d
            # convert to mm/d
            gbp[idx, gbp_vars.index('q_root')] *= 1000
            ax.flatten()[j].plot(gbp[idx, gbp_vars.index('time')],
                                 gbp[idx, gbp_vars.index('q_root')], lw=1, label=scenario)
            depth = f'depth={cp} cm'
            ax.flatten()[j].set(title=depth, xlim=[0, 3650])

    ax[-1, -1].legend()
    ax[0, 0].set(ylabel='Transpiration (mm/d)')
    ax[1, 0].set(ylabel='Transpiration (mm/d)')
    fig.suptitle(site)

    for i in range(4):
        ax[1, i].set(xlabel='Time (d)')

    fig.savefig(f'plots/{site}_VariableTranspiration.png', dpi=300, bbox_inches='tight')

    # Adjust the axis limits and save again
    ax[0, 0].set(xlim=[365, 730])
    fig.savefig(f'plots/{site}_VariableTranspiration_365days.png', dpi=300, bbox_inches='tight')

    # Adjust the axis limits and save again
    ax[0, 0].set(xlim=[365, 455])
    fig.savefig(f'plots/{site}_VariableTranspiration_90days.png', dpi=300, bbox_inches='tight')

    if site != 'Cecil':
        plt.close(fig)

In [ ]:
site = 'Cecil'
for site in tqdm(all_sites):

    fig, ax = plt.subplots(2, 4, figsize=(10, 7), sharex='all', sharey='all')

    for i, scenario in enumerate(all_scenarios):
        sim_folder = Path(f'../min3p_runs/{site}/{scenario}')
        sim_name = scenario

        gbc, gbc_vars, grid_cells = read_min3p_sequence(f'{sim_name}_1.gbc',
                                                        folder=sim_folder, ftype='transient')
        for j, cp in enumerate(control_planes_to_plot):
            grid_cell = 401 - cp # Convert from depth to grid cell number
            idx = list(grid_cells).index(grid_cell)
            ax.flatten()[j].plot(gbc[idx, gbc_vars.index('time')]/365,
                                 gbc[idx, gbc_vars.index('psi03')], lw=1, label=scenario)
            depth = f'depth={cp} cm'
            ax.flatten()[j].set(title=depth, xlim=[0, 3650])

    ax[-1, -1].legend()
    ax[0, 0].set(ylabel='Alk tracer (M)')
    ax[1, 0].set(ylabel='Alk tracer (M)')
    fig.suptitle(site)

    for i in range(4):
        ax[1, i].set(xlabel='Time (y)', xlim=[0, 5])

    fig.savefig(f'plots/{site}_AlkalinityTracerBreakthrough.png', dpi=300, bbox_inches='tight')

    if site != 'Cecil':
        plt.close(fig)

In [ ]:
site = 'Cecil'

for site in tqdm(all_sites):
    fig, ax = plt.subplots(1, 4, figsize=(8, 6), sharey=True)

    scenario = 'monthly'
    sim_folder = Path(f'../min3p_runs/{site}/{scenario}')
    gsc, gsc_cols, timesteps = read_min3p_sequence(f'{scenario}_1.gsc', folder=sim_folder)

    cmap = plt.colormaps['cividis']
    colors = cmap(np.linspace(0, 1, gsc.shape[0]))

    for j, scenario in enumerate(all_scenarios):
        sim_folder = Path(f'../min3p_runs/{site}/{scenario}')
        gsc, gsc_cols, timesteps = read_min3p_sequence(f'{scenario}_1.gsc', folder=sim_folder)
        for i, timestep in enumerate(timesteps):
            z = gsc[i, gsc_cols.index('z'), :]
            depth = z.max() - z  # Convert z (elevation) to depth

            label = '%d d' % round(timestep)
            ax[j].plot(gsc[i, gsc_cols.index('psi03'), :], depth, color=colors[i], label=label)
            ax[j].set(xlabel='Alk tracer (M)', title=scenario)

    ax[0].set(ylabel='Depth', ylim=(depth.max(), depth.min()))

    ax[0].legend()
    fig.suptitle(site)

    fig.savefig(f'plots/{site}_AlkalinityTracerProfiles.png', dpi=300, bbox_inches='tight')

    if site != 'Cecil':
        plt.close(fig)